# 02 - Regles de qualite et corrections

**Projet :** Cadre de gouvernance et qualite des donnees - Hydro-Quebec

**Auteur :** Anthony MISSE

**Date :** 2026-05-10  

Ce notebook formalise et applique les regles de qualite identifiees apres l'exploration initiale.
Les controles sont appliques progressivement sur les deux fichiers sources afin de produire des jeux de donnees nettoyes
et un journal d'anomalies reutilisable dans la suite du projet.

## 1. Import des bibliotheques et chargement

In [17]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_RAW_DIR = Path("../data/raw")
DATA_CLEAN_DIR = Path("../data/clean")
REPORTS_DIR = Path("../reports")

DATA_CLEAN_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [18]:
df_dem = pd.read_csv(DATA_RAW_DIR / 'hq_demande_electricite_raw.csv', encoding='utf-8')
df_sec = pd.read_csv(DATA_RAW_DIR / 'hq_consommation_secteur_raw.csv', encoding='utf-8')

print('hq_demande  :', df_dem.shape)
print('hq_secteur  :', df_sec.shape)

hq_demande  : (43824, 2)
hq_secteur  : (10200, 4)


Les volumes observes dans le notebook 01 sont confirmes. On peut maintenant appliquer les regles de qualite.

## 2. RQ-02 - Format de la colonne `date`

La colonne `date` de `hq_demande` est actuellement de type `object`.
La premiere etape consiste a verifier que toutes les valeurs sont convertibles en horodatage.

In [19]:
date_convertie = pd.to_datetime(df_dem['date'], errors='coerce', utc=True)
nb_invalid_date = date_convertie.isna().sum()
print('Valeurs non convertibles :', nb_invalid_date)
print('Date min :', date_convertie.min())
print('Date max :', date_convertie.max())

Valeurs non convertibles : 0
Date min : 2019-01-01 06:00:00+00:00
Date max : 2024-01-01 05:00:00+00:00


**Conclusion RQ-02 : PASS**

Toutes les valeurs de `date` sont convertibles. La conversion pourra etre integree telle quelle dans le pipeline ADF.

## 3. RQ-03 - Unicite des timestamps


Une serie horaire doit contenir un seul enregistrement par horodatage.
On verifie donc les doublons sur la colonne `date`.

In [20]:
nb_doublons = df_dem.duplicated(subset=['date']).sum()
print('Nombre de doublons sur date :', nb_doublons)

Nombre de doublons sur date : 6


Les doublons identifies doivent etre inspectes avant suppression.

In [21]:
df_doublons = df_dem[df_dem.duplicated(subset=['date'], keep=False)].sort_values('date')
df_doublons

,date,demande (MW)
22954,2019-11-03T01:00:00-05:00,18301.79
22955,2019-11-03T01:00:00-05:00,17963.20
11351,2020-11-01T01:00:00-05:00,20001.77
25860,2020-11-01T01:00:00-05:00,19884.29
28759,2021-11-07T01:00:00-05:00,18793.52
38327,2021-11-07T01:00:00-05:00,18685.38
8169,2022-11-06T01:00:00-05:00,14788.72
8170,2022-11-06T01:00:00-05:00,14447.54
5677,2023-01-01T00:00:00-05:00,22945.84
43551,2023-01-01T00:00:00-05:00,20516.80


Les doublons correspondent majoritairement au changement d'heure de novembre, avec une paire supplementaire au 1er janvier 2023.

In [22]:
df_dem = df_dem.drop_duplicates(subset=['date'], keep='first').copy()
print('Nombre de lignes apres deduplication :', len(df_dem))

Nombre de lignes apres deduplication : 43818


**Conclusion RQ-03 : WARN puis PASS apres correction**

6 lignes dupliquees ont ete detectees puis corrigees par conservation de la premiere occurrence.

## 4. RQ-01 - Completude de `demande_mw`

On controle ensuite la presence de valeurs manquantes dans la mesure de demande electrique.
Le seuil de vigilance retenu est de 1 % des lignes.

In [23]:
nb_null_demande = df_dem['demande (MW)'].isna().sum()
pct_null_demande = round(nb_null_demande * 100 / len(df_dem), 2)
print('Valeurs nulles :', nb_null_demande)
print('Pourcentage   :', pct_null_demande, '%')

Valeurs nulles : 45
Pourcentage   : 0.1 %


Le taux de valeurs manquantes reste tres faible. L'interpolation lineaire est donc acceptable dans ce contexte.

In [24]:
df_dem['demande (MW)'] = df_dem['demande (MW)'].interpolate(method='linear', limit_direction='both')
nb_null_apres = df_dem['demande (MW)'].isna().sum()
print('Valeurs nulles apres interpolation :', nb_null_apres)

Valeurs nulles apres interpolation : 0


**Conclusion RQ-01 : WARN puis PASS apres correction**

45 valeurs nulles ont ete corrigees par interpolation lineaire, sans depasser le seuil de blocage de 1 %.

## 5. RQ-04 - Valeurs extremes de `demande_mw`

Les valeurs tres elevees doivent etre analysees avec prudence : une valeur extreme n'est pas necessairement une erreur.
On applique d'abord la methode IQR, puis un controle technique complementaire.

In [25]:
Q1 = df_dem["demande (MW)"].quantile(0.25)
Q3 = df_dem["demande (MW)"].quantile(0.75)
IQR = Q3 - Q1
seuil_bas = Q1 - 1.5 * IQR
seuil_haut = Q3 + 1.5 * IQR
nb_outliers_iqr = (df_dem["demande (MW)"] > seuil_haut).sum()
nb_outliers_tech = (df_dem["demande (MW)"] > 45000).sum()

print(f"Seuil IQR haut : {seuil_haut:.2f} MW")
print("Valeurs > seuil IQR :", nb_outliers_iqr)
print("Valeurs > 45000 MW :", nb_outliers_tech)

Seuil IQR haut : 37002.65 MW
Valeurs > seuil IQR : 94
Valeurs > 45000 MW : 0


On affiche un extrait des valeurs identifiees pour verifier leur contexte.

In [26]:
df_dem[df_dem['demande (MW)'] > seuil_haut][['date', 'demande (MW)']].head(10)

,date,demande (MW)
233,2023-02-03T09:00:00-05:00,37528.23
234,2023-02-03T13:00:00-05:00,37334.31
235,2023-02-03T14:00:00-05:00,37861.89
236,2023-02-03T17:00:00-05:00,40585.14
237,2023-02-04T03:00:00-05:00,39410.28
238,2023-02-04T05:00:00-05:00,39911.76
240,2023-02-04T19:00:00-05:00,39317.89
5772,2022-01-11T07:00:00-05:00,37844.73
5776,2022-01-11T17:00:00-05:00,38227.06
5777,2022-01-11T20:00:00-05:00,38976.50


**Conclusion RQ-04 : INFO**

94 valeurs depassent le seuil IQR, mais aucune ne depasse 45 000 MW.
Elles sont conservees comme pics legitimes de consommation, probablement lies a des vagues de grand froid.

## 6. RQ-06 - Conformite de `SECTEUR`

La colonne `SECTEUR` doit etre harmonisee pour faciliter les jointures, filtres et visualisations.
On commence par verifier les valeurs brutes puis on applique une normalisation simple.

In [27]:
df_sec['SECTEUR'].value_counts()

INDUSTRIEL        2040
AGRICOLE          2040
COMMERCIAL        2040
INSTITUTIONNEL    2040
RÉSIDENTIEL       2040
Name: SECTEUR, dtype: int64

Les valeurs sont coherentes mais restent en majuscules. On les normalise en format titre.

In [28]:
df_sec['SECTEUR'] = df_sec['SECTEUR'].str.lower().str.title()
df_sec['SECTEUR'].value_counts()

Industriel        2040
Agricole          2040
Commercial        2040
Institutionnel    2040
Résidentiel       2040
Name: SECTEUR, dtype: int64

**Conclusion RQ-06 : PASS apres normalisation**

La distribution reste parfaitement equilibree apres harmonisation des libelles.

## 7. RQ-07 - Validite de `Total (kWh)`

La consommation mensuelle doit etre strictement positive et non nulle.
On verifie la presence de valeurs nulles ou negatives.

In [29]:
nb_null_total = df_sec['Total (kWh)'].isna().sum()
nb_neg_total = (df_sec['Total (kWh)'] <= 0).sum()
print('Valeurs nulles :', nb_null_total)
print('Valeurs <= 0   :', nb_neg_total)

Valeurs nulles : 0
Valeurs <= 0   : 0


**Conclusion RQ-07 : PASS**

Aucune correction n'est necessaire sur `Total (kWh)`.

## 8. Standardisation des noms de colonnes

Les noms de colonnes sources contiennent des espaces, parentheses ou conventions heterogenes.
On applique ici le renommage cible retenu pour les etapes SQL et Power BI.

In [30]:
df_dem = df_dem.rename(columns={"demande (MW)": "demande_mw"})
df_sec = df_sec.rename(
    columns={
        "REGION_ADM_QC_TXT": "region",
        "ANNEE_MOIS": "annee_mois",
        "SECTEUR": "secteur",
        "Total (kWh)": "total_kwh",
    }
)

print(df_dem.columns.tolist())
print(df_sec.columns.tolist())

['date', 'demande_mw']
['region', 'annee_mois', 'secteur', 'total_kwh']


Les deux tables sont maintenant alignees avec une nomenclature simple et exploitable.

## 9. Journal d'anomalies

On construit un fichier central d'anomalies afin de conserver une trace explicite des controles appliques.
Chaque type d'anomalie est documente avec sa severite et son volume.


In [31]:
anomalies = pd.DataFrame(
    [
        {
            "regle": "RQ-01",
            "champ": "demande_mw",
            "type_anomalie": "missing_value",
            "nb_lignes": 45,
            "severite": "WARN",
            "action": "interpolation_lineaire",
        },
        {
            "regle": "RQ-02",
            "champ": "date",
            "type_anomalie": "invalid_datetime",
            "nb_lignes": 0,
            "severite": "PASS",
            "action": "aucune",
        },
        {
            "regle": "RQ-03",
            "champ": "date",
            "type_anomalie": "duplicate_timestamp",
            "nb_lignes": 6,
            "severite": "WARN",
            "action": "keep_first",
        },
        {
            "regle": "RQ-04",
            "champ": "demande_mw",
            "type_anomalie": "outlier_iqr",
            "nb_lignes": 94,
            "severite": "INFO",
            "action": "conserver",
        },
        {
            "regle": "RQ-04",
            "champ": "demande_mw",
            "type_anomalie": "outlier_tech_gt_45000",
            "nb_lignes": 0,
            "severite": "PASS",
            "action": "aucune",
        },
        {
            "regle": "RQ-06",
            "champ": "secteur",
            "type_anomalie": "format_uppercase",
            "nb_lignes": 10200,
            "severite": "INFO",
            "action": "normalisation_title_case",
        },
        {
            "regle": "RQ-07",
            "champ": "total_kwh",
            "type_anomalie": "null_or_non_positive",
            "nb_lignes": 0,
            "severite": "PASS",
            "action": "aucune",
        },
    ]
)
anomalies

,regle,champ,type_anomalie,nb_lignes,severite,action
0,RQ-01,demande_mw,missing_value,45,WARN,interpolation_lineaire
1,RQ-02,date,invalid_datetime,0,PASS,aucune
2,RQ-03,date,duplicate_timestamp,6,WARN,keep_first
3,RQ-04,demande_mw,outlier_iqr,94,INFO,conserver
4,RQ-04,demande_mw,outlier_tech_gt_45000,0,PASS,aucune
5,RQ-06,secteur,format_uppercase,10200,INFO,normalisation_title_case
6,RQ-07,total_kwh,null_or_non_positive,0,PASS,aucune


## 10. Export des jeux nettoyes

In [32]:
anomalies.to_csv(REPORTS_DIR / "anomalies_log.csv", index=False)
df_dem.to_csv(DATA_CLEAN_DIR / "hq_demande_electricite_clean.csv", index=False)
df_sec.to_csv(DATA_CLEAN_DIR / "hq_consommation_secteur_clean.csv", index=False)

resume_regles = pd.DataFrame(
    [
        {
            "regle": "RQ-01",
            "statut": "PASS apres correction",
            "details": "45 nulls interpoles",
        },
        {"regle": "RQ-02", "statut": "PASS", "details": "0 date invalide"},
        {
            "regle": "RQ-03",
            "statut": "PASS apres correction",
            "details": "6 doublons supprimes",
        },
        {"regle": "RQ-04", "statut": "INFO", "details": "94 pics legitimes conserves"},
        {
            "regle": "RQ-06",
            "statut": "PASS apres normalisation",
            "details": "5 secteurs harmonises",
        },
        {"regle": "RQ-07", "statut": "PASS", "details": "0 null et 0 valeur negative"},
    ]
)

resume_regles.to_csv(REPORTS_DIR / "quality_summary.csv", index=False)
print("Exports termines.")

Exports termines.


Les fichiers nettoyes et les rapports de controle sont sauvegardes pour les etapes suivantes du projet.

## 11. Synthese des regles appliquees

| Regle | Objet                            | Resultat                                                          |
|-------|----------------------------------|-------------------------------------------------------------------|
| RQ-01 | Completude de `demande_mw`       | 45 nulls corriges par interpolation, statut PASS apres correction |
| RQ-02 | Convertibilite de `date`         | 0 valeur invalide, statut PASS                                    |
| RQ-03 | Unicite des timestamps           | 6 doublons supprimes, statut PASS apres correction                |
| RQ-04 | Valeurs extremes de `demande_mw` | 94 valeurs > seuil IQR, conservees comme pics legitimes           |
| RQ-06 | Normalisation de `secteur`       | 5 modalites harmonisees, statut PASS                              |
| RQ-07 | Validite de `total_kwh`          | 0 null et 0 valeur <= 0, statut PASS                              |